{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 🗺️ Ethiopian GPS Navigation System\n",
    "## Interactive Demonstration of Dijkstra's Algorithm\n",
    "\n",
    "![Ethiopia Flag](https://upload.wikimedia.org/wikipedia/commons/7/71/Flag_of_Ethiopia.svg)\n",
    "\n",
    "This notebook demonstrates a complete GPS navigation system for Ethiopian roads using Dijkstra's algorithm.\n",
    "\n",
    "### Features Demonstrated:\n",
    "- ✅ Ethiopian city database with real coordinates\n",
    "- ✅ Road network with realistic distances\n",
    "- ✅ Shortest path finding using Dijkstra's algorithm\n",
    "- ✅ Step-by-step algorithm visualization\n",
    "- ✅ Interactive maps and graphs\n",
    "- ✅ Performance comparison of implementations\n",
    "- ✅ Comprehensive reports"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Import required libraries\n",
    "import sys\n",
    "import os\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "# Add project root to path\n",
    "sys.path.append('..')\n",
    "\n",
    "# Import project modules\n",
    "from src.graph.city import City\n",
    "from src.graph.road import Road, RoadType, RoadCondition\n",
    "from src.graph.network import RoadNetwork\n",
    "from src.algorithms.dijkstra_array import DijkstraArray\n",
    "from src.algorithms.dijkstra_pq import DijkstraPriorityQueue\n",
    "from src.algorithms.path_utils import PathUtils\n",
    "from src.utils.data_loader import DataLoader\n",
    "from src.utils.distance_calc import DistanceCalculator\n",
    "from src.visualization.map_plotter import MapPlotter\n",
    "from src.visualization.graph_viz import GraphVisualizer\n",
    "from src.analysis.complexity import ComplexityAnalyzer\n",
    "from src.analysis.report_gen import ReportGenerator\n",
    "\n",
    "print(\"✅ All modules imported successfully!\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Create Ethiopian road network\n",
    "print(\"=\"*60)\n",
    "print(\"📊 CREATING ETHIOPIAN ROAD NETWORK\")\n",
    "print(\"=\"*60)\n",
    "\n",
    "network = RoadNetwork()\n",
    "\n",
    "# Add major Ethiopian cities\n",
    "cities = [\n",
    "    City(1, \"Addis Ababa\", \"Addis Ababa\", 9.03, 38.74, 3500000, 2355, True),\n",
    "    City(2, \"Adama\", \"Oromia\", 8.54, 39.27, 480000, 1712, False),\n",
    "    City(3, \"Bahir Dar\", \"Amhara\", 11.60, 37.38, 350000, 1800, False),\n",
    "    City(4, \"Gondar\", \"Amhara\", 12.60, 37.47, 350000, 2133, False),\n",
    "    City(5, \"Mekelle\", \"Tigray\", 13.50, 39.47, 340000, 2084, False),\n",
    "    City(6, \"Dire Dawa\", \"Dire Dawa\", 9.60, 41.87, 300000, 1276, False),\n",
    "    City(7, \"Jimma\", \"Oromia\", 7.68, 36.83, 250000, 1780, False),\n",
    "    City(8, \"Harar\", \"Harari\", 9.31, 42.13, 150000, 1885, False),\n",
    "    City(9, \"Jijiga\", \"Somali\", 9.35, 42.80, 160000, 1648, False),\n",
    "    City(10, \"Hawassa\", \"SNNPR\", 7.05, 38.47, 350000, 1708, False)\n",
    "]\n",
    "\n",
    "for city in cities:\n",
    "    network.add_city(city)\n",
    "\n",
    "# Add roads between cities\n",
    "roads = [\n",
    "    Road(1, 1, 2, 85, RoadType.HIGHWAY, RoadCondition.GOOD, 100, 4, False, \"Addis-Adama Highway\"),\n",
    "    Road(2, 1, 3, 380, RoadType.HIGHWAY, RoadCondition.GOOD, 100, 4, False, \"Addis-Bahir Highway\"),\n",
    "    Road(3, 2, 3, 380, RoadType.HIGHWAY, RoadCondition.GOOD, 100, 4, False, \"Adama-Bahir Highway\"),\n",
    "    Road(4, 3, 4, 180, RoadType.HIGHWAY, RoadCondition.GOOD, 100, 4, False, \"Bahir-Gondar Highway\"),\n",
    "    Road(5, 4, 5, 380, RoadType.HIGHWAY, RoadCondition.GOOD, 100, 4, False, \"Gondar-Mekelle Highway\"),\n",
    "    Road(6, 1, 5, 783, RoadType.HIGHWAY, RoadCondition.FAIR, 80, 2, False, \"Addis-Mekelle Direct\"),\n",
    "    Road(7, 1, 8, 525, RoadType.HIGHWAY, RoadCondition.FAIR, 80, 2, False, \"Addis-Harar Highway\"),\n",
    "    Road(8, 8, 6, 50, RoadType.REGIONAL, RoadCondition.GOOD, 60, 2, False, \"Harar-Dire Dawa Road\"),\n",
    "    Road(9, 6, 9, 95, RoadType.REGIONAL, RoadCondition.FAIR, 60, 2, False, \"Dire Dawa-Jijiga Road\"),\n",
    "    Road(10, 1, 10, 275, RoadType.HIGHWAY, RoadCondition.GOOD, 100, 4, False, \"Addis-Hawassa Highway\"),\n",
    "    Road(11, 10, 7, 350, RoadType.REGIONAL, RoadCondition.FAIR, 60, 2, False, \"Hawassa-Jimma Road\"),\n",
    "    Road(12, 7, 1, 350, RoadType.HIGHWAY, RoadCondition.GOOD, 80, 2, False, \"Jimma-Addis Highway\")\n",
    "]\n",
    "\n",
    "for road in roads:\n",
    "    network.add_road(road)\n",
    "\n",
    "print(f\"\\n✅ Network created with:\")\n",
    "print(f\"   • {network.get_city_count()} Ethiopian cities\")\n",
    "print(f\"   • {network.get_road_count()} roads\")\n",
    "print(f\"   • Network density: {network.get_density():.4f}\")\n",
    "print(f\"   • Average connections: {network.get_average_degree():.2f}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Display cities\n",
    "print(\"\\n📍 ETHIOPIAN CITIES\")\n",
    "print(\"-\" * 80)\n",
    "print(f\"{'ID':<5} {'City Name':<20} {'Region':<15} {'Population':<15} {'Coordinates'}\")\n",
    "print(\"-\" * 80)\n",
    "\n",
    "for city in network.cities.values():\n",
    "    capital = \" (Capital)\" if city.is_capital else \"\"\n",
    "    print(f\"{city.id:<5} {city.name:<20} {city.region:<15} \"\n",
    "          f\"{city.population/1000000:.1f}M{capital:<10} \"\n",
    "          f\"({city.latitude:.2f}, {city.longitude:.2f})\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Display roads\n",
    "print(\"\\n🛣️  ETHIOPIAN ROADS\")\n",
    "print(\"-\" * 80)\n",
    "print(f\"{'ID':<5} {'From':<15} {'To':<15} {'Distance':<10} {'Type':<10} {'Condition'}\")\n",
    "print(\"-\" * 80)\n",
    "\n",
    "for road in network.roads.values():\n",
    "    city1 = network.get_city_by_id(road.city1_id)\n",
    "    city2 = network.get_city_by_id(road.city2_id)\n",
    "    print(f\"{road.id:<5} {city1.name:<15} {city2.name:<15} \"\n",
    "          f\"{road.distance:<10} {road.road_type.value:<10} {road.condition.value}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Select Addis Ababa as source\n",
    "source_city = network.get_city_by_name(\"Addis Ababa\")\n",
    "if not source_city:\n",
    "    source_city = list(network.cities.values())[0]\n",
    "\n",
    "print(f\"\\n🎯 SOURCE CITY: {source_city.name}\")\n",
    "print(f\"   Region: {source_city.region}\")\n",
    "print(f\"   Coordinates: ({source_city.latitude:.4f}, {source_city.longitude:.4f})\")\n",
    "print(f\"   Population: {source_city.population:,}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import time\n",
    "\n",
    "print(\"\\n🚀 RUNNING DIJKSTRA'S ALGORITHM\")\n",
    "print(\"=\" * 60)\n",
    "\n",
    "# Array implementation\n",
    "print(\"\\n⏱️  Array Implementation (O(V²))...\")\n",
    "array_dijkstra = DijkstraArray(network)\n",
    "start = time.time()\n",
    "distances, parents = array_dijkstra.find_shortest_paths(source_city.id, record_steps=True)\n",
    "array_time = (time.time() - start) * 1000\n",
    "print(f\"   ✅ Completed in {array_time:.2f} ms\")\n",
    "\n",
    "# Priority queue implementation\n",
    "print(\"\\n⏱️  Priority Queue Implementation (O((V+E) log V))...\")\n",
    "pq_dijkstra = DijkstraPriorityQueue(network)\n",
    "start = time.time()\n",
    "pq_distances, pq_parents = pq_dijkstra.find_shortest_paths(source_city.id)\n",
    "pq_time = (time.time() - start) * 1000\n",
    "print(f\"   ✅ Completed in {pq_time:.2f} ms\")\n",
    "\n",
    "# Compare\n",
    "print(f\"\\n📊 PERFORMANCE COMPARISON:\")\n",
    "print(f\"   • Array: {array_time:.2f} ms\")\n",
    "print(f\"   • Priority Queue: {pq_time:.2f} ms\")\n",
    "print(f\"   • Speedup: {array_time/pq_time:.2f}x\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "path_utils = PathUtils(network)\n",
    "\n",
    "print(\"\\n📍 SHORTEST PATHS FROM\", source_city.name)\n",
    "print(\"=\" * 70)\n",
    "\n",
    "# Calculate statistics\n",
    "reachable = [(cid, dist) for cid, dist in distances.items() \n",
    "             if cid != source_city.id and dist != float('inf')]\n",
    "reachable.sort(key=lambda x: x[1])\n",
    "\n",
    "print(f\"\\n📊 Statistics:\")\n",
    "print(f\"   • Total cities: {network.get_city_count()}\")\n",
    "print(f\"   • Reachable: {len(reachable)}\")\n",
    "print(f\"   • Unreachable: {network.get_city_count() - len(reachable) - 1}\")\n",
    "\n",
    "if reachable:\n",
    "    farthest = max(reachable, key=lambda x: x[1])\n",
    "    closest = min(reachable, key=lambda x: x[1])\n",
    "    print(f\"   • Closest: {network.get_city_by_id(closest[0]).name} ({closest[1]:.2f} km)\")\n",
    "    print(f\"   • Farthest: {network.get_city_by_id(farthest[0]).name} ({farthest[1]:.2f} km)\")\n",
    "\n",
    "print(f\"\\n📋 SHORTEST PATHS:\")\n",
    "print(\"-\" * 70)\n",
    "print(f\"{'#':<3} {'Destination':<20} {'Distance':<12} {'Path'}\")\n",
    "print(\"-\" * 70)\n",
    "\n",
    "for i, (city_id, distance) in enumerate(reachable[:8], 1):\n",
    "    city = network.get_city_by_id(city_id)\n",
    "    path_ids = path_utils.reconstruct_path(parents, source_city.id, city_id)\n",
    "    path_str = path_utils.get_path_string(path_ids)\n",
    "    print(f\"{i:<3} {city.name:<20} {distance:<12.2f} {path_str}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"\\n👣 STEP-BY-STEP ALGORITHM DEMONSTRATION\")\n",
    "print(\"=\" * 70)\n",
    "\n",
    "# Show first 5 steps\n",
    "for step in array_dijkstra.step_records[:5]:\n",
    "    print(f\"\\n📌 Step {step['step']}: {step['description']}\")\n",
    "    \n",
    "    # Show current distances\n",
    "    print(\"   Current distances:\")\n",
    "    cities_shown = 0\n",
    "    for city_id, city_idx in network.city_indices.items():\n",
    "        if cities_shown >= 5:\n",
    "            print(\"      ...\")\n",
    "            break\n",
    "        city = network.get_city_by_id(city_id)\n",
    "        dist_val = step['dist'][city_idx]\n",
    "        visited_mark = \"✓\" if step['visited'][city_idx] else \" \"\n",
    "        \n",
    "        if dist_val == float('inf'):\n",
    "            print(f\"      {city.name}[{visited_mark}]: ∞\")\n",
    "        else:\n",
    "            print(f\"      {city.name}[{visited_mark}]: {dist_val:.2f} km\")\n",
    "        cities_shown += 1"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"\\n🗺️  CREATING INTERACTIVE MAP\")\n",
    "print(\"=\" * 60)\n",
    "\n",
    "plotter = MapPlotter(network)\n",
    "\n",
    "# Get path to highlight (Addis to Mekelle)\n",
    "path_ids = path_utils.reconstruct_path(parents, source_city.id, 5)  # Mekelle\n",
    "\n",
    "# Create map\n",
    "map_file = \"ethiopia_network_map.html\"\n",
    "plotter.create_interactive_map(\n",
    "    title=\"Ethiopian Road Network - Shortest Path from Addis Ababa to Mekelle\",\n",
    "    show_cities=True,\n",
    "    show_roads=True,\n",
    "    highlight_path=path_ids,\n",
    "    output_file=map_file\n",
    ")\n",
    "\n",
    "print(f\"\\n✅ Map created: {map_file}\")\n",
    "print(\"📁 Check the file in your file browser and open in web browser\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"\\n📊 CREATING STATIC MAP\")\n",
    "print(\"=\" * 60)\n",
    "\n",
    "plotter.create_static_map(\n",
    "    title=\"Ethiopian Road Network\",\n",
    "    show_cities=True,\n",
    "    show_roads=True,\n",
    "    highlight_path=path_ids,\n",
    "    figsize=(12, 8),\n",
    "    save_path=\"ethiopia_network_static.png\"\n",
    ")\n",
    "\n",
    "print(\"✅ Static map saved as: ethiopia_network_static.png\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"\\n📈 CREATING GRAPH VISUALIZATION\")\n",
    "print(\"=\" * 60)\n",
    "\n",
    "viz = GraphVisualizer(network)\n",
    "\n",
    "# Spring layout\n",
    "viz.set_layout('spring')\n",
    "viz.draw_graph(\n",
    "    title=\"Ethiopian Road Network Graph\",\n",
    "    node_color_by='region',\n",
    "    edge_color_by='type',\n",
    "    show_labels=True,\n",
    "    highlight_path=path_ids,\n",
    "    save_path=\"ethiopia_graph_spring.png\"\n",
    ")\n",
    "\n",
    "print(\"✅ Graph saved as: ethiopia_graph_spring.png\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"\\n🔍 DETAILED PATH ANALYSIS\")\n",
    "print(\"=\" * 60)\n",
    "\n",
    "# Analyze path to Mekelle\n",
    "dest_city = network.get_city_by_id(5)  # Mekelle\n",
    "distance = distances[5]\n",
    "path_ids = path_utils.reconstruct_path(parents, source_city.id, 5)\n",
    "details = path_utils.get_path_details(path_ids)\n",
    "\n",
    "print(f\"\\nPath: {source_city.name} → {dest_city.name}\")\n",
    "print(f\"Total Distance: {distance:.2f} km\")\n",
    "print(f\"\\n📊 Path Statistics:\")\n",
    "print(f\"   • Cities visited: {details['num_cities']}\")\n",
    "print(f\"   • Road segments: {details['num_segments']}\")\n",
    "print(f\"   • Regions visited: {', '.join(details['regions_visited'])}\")\n",
    "print(f\"   • Road types: {', '.join(details['road_types'])}\")\n",
    "\n",
    "print(f\"\\n🗺️  Detailed Path:\")\n",
    "for i, segment in enumerate(details['segments'], 1):\n",
    "    from_city = network.get_city_by_id(segment['from_id'])\n",
    "    to_city = network.get_city_by_id(segment['to_id'])\n",
    "    print(f\"   {i}. {from_city.name} → {to_city.name}: {segment['distance']} km ({segment['road_type']})\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"\\n⏱️  COMPLEXITY ANALYSIS\")\n",
    "print(\"=\" * 60)\n",
    "\n",
    "analyzer = ComplexityAnalyzer(network)\n",
    "analysis = analyzer.analyze_theoretical()\n",
    "\n",
    "V = analysis['vertices']\n",
    "E = analysis['edges']\n",
    "\n",
    "print(f\"\\n📊 Graph Statistics:\")\n",
    "print(f\"   • Vertices (V): {V}\")\n",
    "print(f\"   • Edges (E): {E}\")\n",
    "print(f\"   • Density: {analysis['graph_density']:.4f}\")\n",
    "print(f\"   • Graph type: {analysis['graph_type'].upper()}\")\n",
    "\n",
    "print(f\"\\n⏱️  Time Complexity:\")\n",
    "print(f\"   • Array Implementation: O(V²) = O({V}²) = {V*V:,} operations\")\n",
    "print(f\"   • Priority Queue: O((V+E) log V) = {analysis['pq_implementation']['time_operations']:,} ops\")\n",
    "print(f\"   • Theoretical Speedup: {analysis['comparison']['speedup_theoretical']:.2f}x\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"\\n📑 GENERATING REPORT\")\n",
    "print(\"=\" * 60)\n",
    "\n",
    "reporter = ReportGenerator(network, source_city.id)\n",
    "\n",
    "# Generate text report preview\n",
    "print(\"\\n📄 Report Preview:\")\n",
    "text_report = reporter.generate_text_report(include_all=False)\n",
    "print(text_report[:500] + \"...\")\n",
    "\n",
    "# Save full report\n",
    "with open(\"ethiopia_gps_report.txt\", 'w', encoding='utf-8') as f:\n",
    "    f.write(text_report)\n",
    "\n",
    "print(\"\\n✅ Full report saved to: ethiopia_gps_report.txt\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"\\n🎮 INTERACTIVE PATH FINDER\")\n",
    "print(\"=\" * 60)\n",
    "\n",
    "def find_path(city1_name, city2_name):\n",
    "    \"\"\"Find path between two cities\"\"\"\n",
    "    city1 = network.get_city_by_name(city1_name)\n",
    "    city2 = network.get_city_by_name(city2_name)\n",
    "    \n",
    "    if not city1 or not city2:\n",
    "        return \"❌ City not found\"\n",
    "    \n",
    "    dist, path = pq_dijkstra.find_shortest_path_to(city1.id, city2.id)\n",
    "    \n",
    "    if dist == float('inf'):\n",
    "        return f\"❌ No path found from {city1.name} to {city2.name}\"\n",
    "    \n",
    "    path_names = [network.get_city_by_id(cid).name for cid in path]\n",
    "    return f\"✅ Distance: {dist:.2f} km\\n   Path: {' → '.join(path_names)}\"\n",
    "\n",
    "# Example queries\n",
    "print(find_path(\"Addis Ababa\", \"Mekelle\"))\n",
    "print(\"\\n\" + \"-\"*40)\n",
    "print(find_path(\"Bahir Dar\", \"Gondar\"))\n",
    "print(\"\\n\" + \"-\"*40)\n",
    "print(find_path(\"Dire Dawa\", \"Harar\"))\n",
    "\n",
    "# Try your own query\n",
    "print(\"\\n\" + \"=\"*40)\n",
    "city1 = input(\"Enter first city: \").strip()\n",
    "city2 = input(\"Enter second city: \").strip()\n",
    "print(\"\\n\" + find_path(city1, city2))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"\\n📊 PERFORMANCE VISUALIZATION\")\n",
    "print(\"=\" * 60)\n",
    "\n",
    "# Create scalability test\n",
    "sizes = [10, 20, 30, 40, 50]\n",
    "array_times = []\n",
    "pq_times = []\n",
    "\n",
    "for size in sizes:\n",
    "    test_network = RoadNetwork()\n",
    "    test_network.generate_ethiopian_network(num_cities=size, num_roads=size*2)\n",
    "    source = list(test_network.cities.values())[0]\n",
    "    \n",
    "    # Test array\n",
    "    start = time.time()\n",
    "    DijkstraArray(test_network).find_shortest_paths(source.id)\n",
    "    array_times.append((time.time() - start) * 1000)\n",
    "    \n",
    "    # Test PQ\n",
    "    start = time.time()\n",
    "    DijkstraPriorityQueue(test_network).find_shortest_paths(source.id)\n",
    "    pq_times.append((time.time() - start) * 1000)\n",
    "\n",
    "# Plot\n",
    "plt.figure(figsize=(10, 6))\n",
    "plt.plot(sizes, array_times, 'b-o', label='Array O(V²)', linewidth=2)\n",
    "plt.plot(sizes, pq_times, 'r-o', label='Priority Queue O((V+E) log V)', linewidth=2)\n",
    "plt.xlabel('Number of Cities (V)', fontsize=12)\n",
    "plt.ylabel('Time (ms)', fontsize=12)\n",
    "plt.title('Algorithm Scalability Comparison', fontsize=14, fontweight='bold')\n",
    "plt.legend(fontsize=10)\n",
    "plt.grid(True, alpha=0.3)\n",
    "plt.tight_layout()\n",
    "plt.savefig('performance_comparison.png', dpi=300)\n",
    "plt.show()\n",
    "\n",
    "print(\"✅ Performance graph saved as: performance_comparison.png\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"\\n\" + \"=\"*70)\n",
    "print(\"✅ DEMONSTRATION COMPLETE\")\n",
    "print(\"=\"*70)\n",
    "\n",
    "print(f\"\"\"\n",
    "📊 SUMMARY:\n",
    "----------\n",
    "• Network: {network.get_city_count()} Ethiopian cities, {network.get_road_count()} roads\n",
    "• Source City: {source_city.name}\n",
    "• Algorithm Performance: {array_time:.2f}ms (Array) vs {pq_time:.2f}ms (PQ)\n",
    "• Speedup: {array_time/pq_time:.2f}x\n",
    "\n",
    "📁 FILES GENERATED:\n",
    "-----------------\n",
    "1. 🗺️  Interactive Map: ethiopia_network_map.html\n",
    "2. 📊 Static Map: ethiopia_network_static.png\n",
    "3. 📈 Graph: ethiopia_graph_spring.png\n",
    "4. 📄 Report: ethiopia_gps_report.txt\n",
    "5. 📊 Performance Chart: performance_comparison.png\n",
    "\n",
    "🏁 To view the interactive map, open ethiopia_network_map.html in your browser!\n",
    "\"\"\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.9.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}